# Laboratorio: vectores, geometría de $\mathbb R^n$, matrices y sistemas

Este cuaderno acompaña las clases C1 y C2. La teoría y las demostraciones se encuentran en la hoja de lectura. Aquí vamos a calcular, visualizar, formular conjeturas y comprobar propiedades.

Al finalizar deberías poder interpretar cada resultado, no solamente obtenerlo con Python.

In [ ]:
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## C1. Vectores y geometría en $\mathbb R^n$

Comenzamos comparando operaciones implementadas directamente con listas y las operaciones de `numpy`.

In [ ]:
def sumar_vectores(u, v):
    if len(u) != len(v):
        raise ValueError("Los vectores deben tener la misma dimensión")
    return [ui + vi for ui, vi in zip(u, v)]

def multiplicar_por_escalar(alpha, u):
    return [alpha * ui for ui in u]

u_lista = [2, -1, 3]
v_lista = [1, 4, -2]

print("Con listas:", sumar_vectores(u_lista, v_lista))
print("Con listas:", multiplicar_por_escalar(-2, u_lista))

u = np.array(u_lista, dtype=float)
v = np.array(v_lista, dtype=float)
print("Con NumPy:", u + v)
print("Con NumPy:", -2 * u)

### Visualización de suma y múltiplos

En el plano, la suma puede interpretarse mediante la regla del paralelogramo.

In [ ]:
def dibujar_vectores(vectores, etiquetas, colores=None, titulo="Vectores"):
    colores = colores or [None] * len(vectores)
    fig, ax = plt.subplots(figsize=(6, 6))
    for vector, etiqueta, color in zip(vectores, etiquetas, colores):
        ax.quiver(0, 0, vector[0], vector[1], angles="xy",
                  scale_units="xy", scale=1, color=color, label=etiqueta)
    maximo = max(1, max(np.max(np.abs(w)) for w in vectores)) + 1
    ax.set_xlim(-maximo, maximo)
    ax.set_ylim(-maximo, maximo)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title(titulo)
    plt.show()

a = np.array([2.0, 1.0])
b = np.array([-1.0, 2.0])
def dibujar_suma_cabeza_cola(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    suma = a + b
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.quiver(0, 0, a[0], a[1], angles="xy", scale_units="xy", scale=1,
              color="tab:blue", label="a")
    ax.quiver(a[0], a[1], b[0], b[1], angles="xy", scale_units="xy", scale=1,
              color="tab:orange", label="b trasladado")
    ax.quiver(0, 0, suma[0], suma[1], angles="xy", scale_units="xy", scale=1,
              color="tab:green", label="a+b")
    ax.plot([0, b[0], suma[0]], [0, b[1], suma[1]],
            color="gray", linestyle="--", alpha=0.6)
    maximo = max(1, np.max(np.abs(np.concatenate([a, b, suma])))) + 1
    ax.set_xlim(-maximo, maximo)
    ax.set_ylim(-maximo, maximo)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title("Suma mediante la regla cabeza-cola")
    plt.show()

dibujar_suma_cabeza_cola(a, b)
dibujar_vectores([a, 2*a, -1.5*a], ["a", "2a", "-1.5a"],
                  ["tab:blue", "tab:green", "tab:red"],
                  "Múltiplos de un vector")

### Norma, distancia y normalización

In [ ]:
def norma_sin_numpy(x):
    return math.sqrt(sum(xi**2 for xi in x))

def normalizar(x, tolerancia=1e-12):
    x = np.asarray(x, dtype=float)
    norma = np.linalg.norm(x)
    if norma < tolerancia:
        raise ValueError("El vector cero no se puede normalizar")
    return x / norma

x = np.array([3.0, 4.0])
y = np.array([-1.0, 2.0])

print("Norma sin NumPy:", norma_sin_numpy(x))
print("Norma con NumPy:", np.linalg.norm(x))
print("Distancia entre x e y:", np.linalg.norm(x - y))
print("Vector unitario en la dirección de x:", normalizar(x))
print("Norma del vector unitario:", np.linalg.norm(normalizar(x)))

### Producto punto y Cauchy-Schwarz

Para todo $u,v\in\mathbb R^n$,

$$|u^Tv|\leq \|u\|\,\|v\|.$$

A continuación calcularemos ambos lados en varios ejemplos. Estas comprobaciones numéricas ilustran el teorema, pero no constituyen una prueba. La prueba se encuentra en la hoja teórica.

In [ ]:
def producto_punto_sin_numpy(u, v):
    if len(u) != len(v):
        raise ValueError("Los vectores deben tener la misma dimensión")
    return sum(ui * vi for ui, vi in zip(u, v))

u_prueba = [1, 2, -1]
v_prueba = [3, 0, 2]
print("Producto punto sin NumPy:", producto_punto_sin_numpy(u_prueba, v_prueba))
print("Producto punto con NumPy:", np.array(u_prueba) @ np.array(v_prueba))

def verificar_cauchy_schwarz(u, v, tolerancia=1e-12):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    izquierda = abs(u @ v)
    derecha = np.linalg.norm(u) * np.linalg.norm(v)
    return izquierda, derecha, izquierda <= derecha + tolerancia

pares = [
    (np.array([1, 2, -1]), np.array([3, 0, 2])),
    (np.array([1, 2, 3]), np.array([2, 4, 6])),
    (np.array([1, 0]), np.array([0, 1])),
]

for u, v in pares:
    izquierda, derecha, se_cumple = verificar_cauchy_schwarz(u, v)
    print(f"u={u}, v={v}: {izquierda:.4f} <= {derecha:.4f}: {se_cumple}")

### Ángulo y ortogonalidad

El recorte numérico con `np.clip` evita que los errores de redondeo produzcan un valor ligeramente fuera del intervalo $[-1,1]$.

In [ ]:
def angulo_entre_vectores(u, v, grados=True):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    denominador = np.linalg.norm(u) * np.linalg.norm(v)
    if np.isclose(denominador, 0):
        raise ValueError("El ángulo no está definido para el vector cero")
    coseno = np.clip((u @ v) / denominador, -1.0, 1.0)
    angulo = np.arccos(coseno)
    return np.degrees(angulo) if grados else angulo

ejemplos = [
    (np.array([1, 0]), np.array([1, 1])),
    (np.array([1, 2]), np.array([-2, 1])),
    (np.array([1, 0]), np.array([-1, 1])),
]

for u, v in ejemplos:
    print(f"Ángulo entre {u} y {v}: {angulo_entre_vectores(u, v):.2f} grados")

### Actividades de C1

1. Modifica los vectores de los ejemplos y predice el signo de su producto punto antes de ejecutar.
2. Busca dos vectores distintos para los cuales haya igualdad en Cauchy-Schwarz. Explica por qué.
3. Construye dos vectores no nulos ortogonales en $\mathbb R^3$.
4. Comprueba numéricamente la desigualdad triangular para cinco pares aleatorios.
5. Explica por qué el ángulo con el vector cero no está definido.

## C2. Matrices y sistemas sencillos

### Propiedades algebraicas y dimensiones

Usaremos matrices conformables para comprobar las dos distributividades, la asociatividad del producto y la compatibilidad con escalares. Una comprobación numérica no sustituye una prueba, pero ayuda a detectar qué expresiones están definidas y qué tamaños producen.

In [ ]:
A = np.array([[1., 2., 0.],
              [0., -1., 3.]])       # 2 x 3
D = np.array([[2., 0., 1.],
              [-1., 4., 2.]])       # 2 x 3
B = np.array([[2., 1.],
              [1., 0.],
              [-1., 4.]])           # 3 x 2
C = np.array([[0., 2.],
              [3., -1.],
              [1., 1.]])            # 3 x 2
R = np.array([[1., 2.],
              [0., -1.]])           # 2 x 2
alpha = -3

print("formas: A", A.shape, ", B", B.shape, ", AB", (A @ B).shape)
print("A(B+C) = AB+AC:", np.allclose(A @ (B + C), A @ B + A @ C))
print("(A+D)B = AB+DB:", np.allclose((A + D) @ B, A @ B + D @ B))
print("(AB)R = A(BR):", np.allclose((A @ B) @ R, A @ (B @ R)))
print("alpha(AB) = (alpha A)B:",
      np.allclose(alpha * (A @ B), (alpha * A) @ B))

### El producto matricial no es conmutativo

Reproducimos el ejemplo de la hoja teórica. Aunque ambos productos existen y son matrices $2	imes2$, sus resultados son distintos.

In [ ]:
P = np.array([[1., 1.],
              [0., 1.]])
Q = np.array([[1., 0.],
              [1., 1.]])

print("PQ =\n", P @ Q)
print("QP =\n", Q @ P)
print("¿PQ=QP?", np.allclose(P @ Q, Q @ P))

### Identidad, transposición e inversa

Para $A\in\mathbb R^{m	imes n}$, la identidad de la izquierda es $I_m$ y la de la derecha es $I_n$. También comprobamos las cuatro propiedades de la transposición. Finalmente separamos la verificación de una inversa de su aplicación a un sistema lineal.

In [ ]:
m, n = A.shape
I_m = np.eye(m)
I_n = np.eye(n)

print("I_m tiene forma", I_m.shape, "e I_n tiene forma", I_n.shape)
print("I_m A = A:", np.allclose(I_m @ A, A))
print("A I_n = A:", np.allclose(A @ I_n, A))

print("\nPropiedades de la transposición")
print("(A^T)^T = A:", np.allclose(A.T.T, A))
print("(A+D)^T = A^T+D^T:", np.allclose((A + D).T, A.T + D.T))
print("(alpha A)^T = alpha A^T:", np.allclose((alpha * A).T, alpha * A.T))
print("(AB)^T = B^T A^T:", np.allclose((A @ B).T, B.T @ A.T))

S = np.array([[1., 2.], [3., 4.]])
S_inv = np.linalg.inv(S)
I_2 = np.eye(2)
print("\nS^{-1} =\n", S_inv)
print("S S^{-1} = I:", np.allclose(S @ S_inv, I_2))
print("S^{-1} S = I:", np.allclose(S_inv @ S, I_2))

b_sistema = np.array([5., 11.])
x_por_inversa = S_inv @ b_sistema
x_por_solve = np.linalg.solve(S, b_sistema)
print("solución mediante S^{-1}b:", x_por_inversa)
print("coincide con solve:", np.allclose(x_por_inversa, x_por_solve))
print("residuo b-Sx:", b_sistema - S @ x_por_inversa)

### Multiplicación por bloques

Construimos particiones conformables con tamaños internos $2$ y $1$. Primero calculamos los cuatro bloques del producto y después comprobamos que ensamblarlos produce la misma matriz que la multiplicación ordinaria.

In [ ]:
A11 = np.array([[1., 2.], [0., 1.]])   # 2 x 2
A12 = np.array([[3.], [4.]])               # 2 x 1
A21 = np.array([[2., -1.]])                 # 1 x 2
A22 = np.array([[5.]])                      # 1 x 1

B11 = np.array([[1.], [2.]])                # 2 x 1
B12 = np.array([[0., 1.], [1., 0.]])        # 2 x 2
B21 = np.array([[3.]])                      # 1 x 1
B22 = np.array([[2., -1.]])                 # 1 x 2

M = np.block([[A11, A12], [A21, A22]])
N = np.block([[B11, B12], [B21, B22]])

C11 = A11 @ B11 + A12 @ B21
C12 = A11 @ B12 + A12 @ B22
C21 = A21 @ B11 + A22 @ B21
C22 = A21 @ B12 + A22 @ B22
C_por_bloques = np.block([[C11, C12], [C21, C22]])

print("M =\n", M)
print("N =\n", N)
print("producto ensamblado por bloques =\n", C_por_bloques)
print("¿Coincide con M@N?", np.allclose(C_por_bloques, M @ N))
print("formas de C11, C12, C21, C22:",
      C11.shape, C12.shape, C21.shape, C22.shape)

### $Ax$ como combinación de las columnas de $A$

In [ ]:
M_columnas = np.array([[1., 0., 2.],
              [2., 1., -1.],
              [0., 3., 1.]])
x = np.array([2., -1., 3.])

producto = M_columnas @ x
combinacion = x[0]*M_columnas[:, 0] + x[1]*M_columnas[:, 1] + x[2]*M_columnas[:, 2]

print("M_columnas x =", producto)
print("Combinación de columnas =", combinacion)
print("¿Son iguales?", np.allclose(producto, combinacion))

### Matrices y sistemas diagonales

`np.diag([d1, ..., dn])` construye la matriz $\operatorname{diag}(d_1,\ldots,d_n)$. Cuando sus entradas diagonales son no nulas, cada ecuación del sistema se resuelve independientemente mediante $x_i=b_i/d_i$. Conservamos dos perspectivas: cálculo vectorizado y recorrido explícito con fracciones.

In [ ]:
def resolver_diagonal(D, b):
    D = np.asarray(D, dtype=float)
    b = np.asarray(b, dtype=float)
    if D.ndim != 2 or D.shape[0] != D.shape[1] or D.shape[0] != len(b):
        raise ValueError("Las dimensiones de D y b no son compatibles")
    if not np.allclose(D, np.diag(np.diag(D))):
        raise ValueError("D debe ser diagonal")
    diagonal = np.diag(D)
    if np.any(np.isclose(diagonal, 0)):
        raise ValueError("Este procedimiento requiere diagonal no nula")
    return b / diagonal

D_1 = np.diag([2., 5., -3.])
b_1 = np.array([4., 10., -6.])
x_1 = resolver_diagonal(D_1, b_1)
print("Primer sistema:")
print("x =", x_1)
print("residuo b-Dx =", b_1 - D_1 @ x_1)

D_2 = np.diag([3., 2., -1.])
b_2 = np.array([7., 8., 4.])
x_2 = np.zeros(len(b_2))
for i in range(len(b_2)):
    x_2[i] = b_2[i] / D_2[i, i]

print("\nSegundo sistema, expresado con fracciones:")
print([str(Fraction(valor).limit_denominator()) for valor in x_2])
print("residuo b-Dx =", b_2 - D_2 @ x_2)

### Matrices y sistemas triangulares

`np.triu` conserva la parte triangular superior y `np.tril` la inferior. Las funciones siguientes verifican primero la forma de la matriz y luego aplican las fórmulas recursivas de sustitución. Además de la solución, devuelven un conteo elemental de multiplicaciones, sumas/restas y divisiones.

In [ ]:
def sustitucion_adelante(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    if L.ndim != 2 or L.shape[0] != L.shape[1] or L.shape[0] != len(b):
        raise ValueError("Las dimensiones de L y b no son compatibles")
    if not np.allclose(L, np.tril(L)):
        raise ValueError("L debe ser triangular inferior")
    n = len(b)
    x = np.zeros(n)
    operaciones = 0
    for i in range(n):
        if np.isclose(L[i, i], 0):
            raise ValueError(f"Pivote diagonal nulo en la posición {i}")
        suma = 0.0
        for j in range(i):
            suma += L[i, j] * x[j]
            operaciones += 2
        x[i] = (b[i] - suma) / L[i, i]
        operaciones += 2
    return x, operaciones


def sustitucion_atras(U, b):
    U = np.asarray(U, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    if U.ndim != 2 or U.shape[0] != U.shape[1] or U.shape[0] != len(b):
        raise ValueError("Las dimensiones de U y b no son compatibles")
    if not np.allclose(U, np.triu(U)):
        raise ValueError("U debe ser triangular superior")
    n = len(b)
    x = np.zeros(n)
    operaciones = 0
    for i in range(n - 1, -1, -1):
        if np.isclose(U[i, i], 0):
            raise ValueError(f"Pivote diagonal nulo en la posición {i}")
        suma = 0.0
        for j in range(i + 1, n):
            suma += U[i, j] * x[j]
            operaciones += 2
        x[i] = (b[i] - suma) / U[i, i]
        operaciones += 2
    return x, operaciones


L = np.array([[2., 0., 0.], [1., 3., 0.], [4., 2., -1.]])
b_L = np.array([4., 5., -4.])
x_L, ops_L = sustitucion_adelante(L, b_L)

U = np.array([[2., -1., 3.], [0., 4., 2.], [0., 0., 5.]])
b_U = np.array([9., 10., 15.])
x_U, ops_U = sustitucion_atras(U, b_U)

print("L es triangular inferior:", np.allclose(L, np.tril(L)))
print("U es triangular superior:", np.allclose(U, np.triu(U)))
print("\nSustitución hacia adelante")
print("x =", x_L, "; operaciones =", ops_L)
print("residuo b-Lx =", b_L - L @ x_L)
print("\nSustitución hacia atrás")
print("x =", x_U, "; operaciones =", ops_U)
print("residuo b-Ux =", b_U - U @ x_U)

### Actividades de C2

1. Antes de ejecutar un producto, escribe las dimensiones de los factores y del resultado.
2. Comprueba las dos distributividades y la asociatividad con otras matrices conformables.
3. Construye matrices rectangulares para las que $AB$ esté definido, pero $BA$ no.
4. Encuentra dos matrices cuadradas distintas de las del ejemplo para las que $AB
eq BA$.
5. Cambia los bloques del ejemplo, conserva sus tamaños y verifica nuevamente la fórmula del producto por bloques.
6. Para una matriz $A\in\mathbb R^{4	imes3}$, construye las identidades correctas y comprueba $I_4A=A=AI_3$.
7. Verifica numéricamente las cuatro propiedades de la transposición con nuevos datos.
8. Intenta invertir una matriz singular, registra el mensaje de Python y explica su significado matemático.
9. Modifica `resolver_diagonal` para distinguir entre una ecuación incompatible y una variable libre cuando aparece un cero en la diagonal.
10. Cambia los sistemas triangulares, verifica siempre el residuo y compara el conteo de operaciones para órdenes 3, 10 y 100.
11. Escoge una matriz y un vector compatibles y escribe manualmente $Ax$ como combinación de las columnas antes de comprobarlo con Python.

La próxima clase introducirá las operaciones elementales sobre ecuaciones y las sistematizará mediante eliminación de Gauss–Jordan.